In [1]:
import sqlite3
import pandas as pd

In [2]:
#Check

import sys
print(sys.executable)

c:\Users\VIOLLETTE MONCHARI\.conda\envs\dsml\python.exe


In [4]:
import pandas as pd
print(pd.__version__)

2.3.3


In [ ]:
#Connect to the DB
import sqlite3
import pandas as pd

db_path = "../sql/ecommerce.db"
conn = sqlite3.connect(db_path)

print("Connected to:", db_path)

Connected to: ../sql/ecommerce.db


In [6]:
#Load the cleaned CSV into pandas

df_clean = pd.read_csv("../data/cleaned/online_retail_cleaned.csv")
invoice_summary = pd.read_csv("../data/cleaned/invoice_summary.csv")
customer_summary = pd.read_csv("../data/cleaned/customer_summary.csv")

df_clean.shape, invoice_summary.shape, customer_summary.shape

C:\Users\VIOLLETTE MONCHARI\AppData\Local\Temp\ipykernel_18192\2000160187.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv("../data/cleaned/online_retail_cleaned.csv")


((530104, 9), (19960, 6), (4338, 5))

In [7]:
#Write Tables into SQLlite

df_clean.to_sql("transactions", conn, if_exists="replace", index=False)
invoice_summary.to_sql("invoices", conn, if_exists="replace", index=False)
customer_summary.to_sql("customers", conn, if_exists="replace", index=False)

print("Tables written: transactions, invoices, customers")

Tables written: transactions, invoices, customers


In [8]:
#Verify Tables Exist

pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,transactions
1,invoices
2,customers


In [9]:
#Confirm what tables exist

query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name
0,transactions
1,invoices
2,customers


In [10]:
#Look at the data using SQL not pandas

query = """
SELECT *
FROM transactions
LIMIT 5;
"""

pd.read_sql(query, conn)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineRevenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [11]:
#Count Rows

query = """
SELECT COUNT(*) AS total_rows
FROM transactions;
"""

pd.read_sql(query, conn)

,total_rows
0,530104


In [12]:
#Total Revenue in the dataset

query = """
SELECT SUM(LineRevenue) AS total_revenue
FROM transactions;
"""

pd.read_sql(query, conn)

,total_revenue
0,1.066668e+07


In [13]:
#Revenue per country

query = """
SELECT
    Country,
    SUM(LineRevenue) AS country_revenue
FROM transactions
GROUP BY Country
ORDER BY country_revenue DESC;
"""

pd.read_sql(query, conn)

,Country,country_revenue
0,United Kingdom,9025222.084
1,Netherlands,285446.340
2,EIRE,283453.960
3,Germany,228867.140
4,France,209715.110
5,Australia,138521.310
6,Spain,61577.110
7,Switzerland,57089.900
8,Belgium,41196.340
9,Sweden,38378.330


In [14]:
#Which products bring in the mmost money

query = """
SELECT
    Description,
    SUM(LineRevenue) AS total_revenue
FROM transactions
GROUP BY Description
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,Description,total_revenue
0,DOTCOM POSTAGE,206248.77
1,REGENCY CAKESTAND 3 TIER,174484.74
2,"PAPER CRAFT , LITTLE BIRDIE",168469.60
3,WHITE HANGING HEART T-LIGHT HOLDER,106292.77
4,PARTY BUNTING,99504.33
5,JUMBO BAG RED RETROSPOT,94340.05
6,MEDIUM CERAMIC TOP STORAGE JAR,81700.92
7,Manual,78112.82
8,POSTAGE,78101.88
9,RABBIT NIGHT LIGHT,66964.99


In [15]:
#Who are our most valuable customers

query = """
SELECT
    CustomerID,
    SUM(LineRevenue) AS customer_revenue
FROM transactions
GROUP BY CustomerID
ORDER BY customer_revenue DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,CustomerID,customer_revenue
0,NaN,1755276.64
1,14646.0,280206.02
2,18102.0,259657.30
3,17450.0,194550.79
4,16446.0,168472.50
5,14911.0,143825.06
6,12415.0,124914.53
7,14156.0,117379.63
8,17511.0,91062.38
9,16029.0,81024.84


In [16]:
#Where is our money coming from Geographically

query = """
SELECT
    Country,
    SUM(LineRevenue) AS total_revenue
FROM transactions
GROUP BY Country
ORDER BY total_revenue DESC;
"""

pd.read_sql(query, conn)


,Country,total_revenue
0,United Kingdom,9025222.084
1,Netherlands,285446.340
2,EIRE,283453.960
3,Germany,228867.140
4,France,209715.110
5,Australia,138521.310
6,Spain,61577.110
7,Switzerland,57089.900
8,Belgium,41196.340
9,Sweden,38378.330


In [17]:
#How many customers bought only once vs multiple times
# Use sql to aggregate

query = """
SELECT
    CustomerID,
    COUNT(DISTINCT InvoiceNo) AS num_invoices
FROM transactions
GROUP BY CustomerID;
"""

customer_invoices = pd.read_sql(query, conn)
customer_invoices.head()

,CustomerID,num_invoices
0,NaN,1428
1,12346.0,1
2,12347.0,7
3,12348.0,4
4,12349.0,1


In [18]:
#use python for labelling/logic

customer_invoices['customer_type'] = customer_invoices['num_invoices'].apply(
    lambda x: 'one-time' if x == 1 else 'repeat'
)

customer_invoices['customer_type'].value_counts()

customer_type
repeat      2846
one-time    1493
Name: count, dtype: int64

In [19]:
#Save top products by revenue

top_products = pd.read_sql("""
SELECT
    Description,
    SUM(LineRevenue) AS total_revenue
FROM transactions
GROUP BY Description
ORDER BY total_revenue DESC
LIMIT 10;
""", conn)

top_products.to_csv("../data/cleaned/sql_top_products.csv", index=False)
top_products.head(10)

,Description,total_revenue
0,DOTCOM POSTAGE,206248.77
1,REGENCY CAKESTAND 3 TIER,174484.74
2,"PAPER CRAFT , LITTLE BIRDIE",168469.60
3,WHITE HANGING HEART T-LIGHT HOLDER,106292.77
4,PARTY BUNTING,99504.33
5,JUMBO BAG RED RETROSPOT,94340.05
6,MEDIUM CERAMIC TOP STORAGE JAR,81700.92
7,Manual,78112.82
8,POSTAGE,78101.88
9,RABBIT NIGHT LIGHT,66964.99


In [20]:
#Save top customers by revenue

top_customers_sql = pd.read_sql("""
SELECT
    CustomerID,
    SUM(LineRevenue) AS customer_revenue
FROM transactions
WHERE CustomerID IS NOT NULL
GROUP BY CustomerID
ORDER BY customer_revenue DESC
LIMIT 10;
""", conn)

top_customers_sql.to_csv("../data/cleaned/sql_top_customers.csv", index=False)
top_customers_sql.head(10)

,CustomerID,customer_revenue
0,14646.0,280206.02
1,18102.0,259657.30
2,17450.0,194550.79
3,16446.0,168472.50
4,14911.0,143825.06
5,12415.0,124914.53
6,14156.0,117379.63
7,17511.0,91062.38
8,16029.0,81024.84
9,12346.0,77183.60


In [21]:
#Save revenue by country

country_revenue_sql = pd.read_sql("""
SELECT
    Country,
    SUM(LineRevenue) AS country_revenue
FROM transactions
GROUP BY Country
ORDER BY country_revenue DESC;
""", conn)

country_revenue_sql.to_csv("../data/cleaned/sql_country_revenue.csv", index=False)
country_revenue_sql.head(10)

,Country,country_revenue
0,United Kingdom,9025222.084
1,Netherlands,285446.340
2,EIRE,283453.960
3,Germany,228867.140
4,France,209715.110
5,Australia,138521.310
6,Spain,61577.110
7,Switzerland,57089.900
8,Belgium,41196.340
9,Sweden,38378.330


In [22]:
#Save one time vs repeat customers

customer_invoices_sql = pd.read_sql("""
SELECT
    CustomerID,
    COUNT(DISTINCT InvoiceNo) AS num_invoices
FROM transactions
WHERE CustomerID IS NOT NULL
GROUP BY CustomerID;
""", conn)

customer_invoices_sql["customer_type"] = customer_invoices_sql["num_invoices"].apply(
    lambda x: "one-time" if x == 1 else "repeat"
)

customer_type_counts = customer_invoices_sql["customer_type"].value_counts().reset_index()
customer_type_counts.columns = ["customer_type", "count"]

customer_invoices_sql.to_csv("../data/cleaned/sql_customer_invoices.csv", index=False)
customer_type_counts.to_csv("../data/cleaned/sql_customer_type_counts.csv", index=False)

customer_type_counts

,customer_type,count
0,repeat,2845
1,one-time,1493


In [23]:
#Close db connection

conn.close()
print("Database connection closed.")

Database connection closed.
